---
title: "ESCUELA POLITECNICA NACIONAL"
subtitle: "PRACTICA #2: Practica Muestra vs Poblacion"
author: "Aidan Carrasco"
date: today
lang: es
format: 
  pdf:
    papersize: letter
    margin-left: 2.5cm
    margin-right: 2.5cm
    margin-top: 3cm
    margin-bottom: 3cm
header-includes:
- \usepackage{fancyhdr}
- \pagestyle{fancy}
- \fancyhead[L]{Ciencia de Datos e Inteligencia Artificial}
- \fancyhead[R]{EPN}
- \fancyfoot[C]{\thepage} 
---

| | |
|---|---|
| **Estudiante** | Aidan Carrasco — usuario de GitHub: [0-AHCA-0] |
| **Carrera** | Ciencia de Datos e Inteligencia Artificial |
| **Docente** | CHRISTIAN GONZALO SUAREZ COBOS |
| **Período académico** | 2026-B |
| **Fecha de entrega** | 19/10/2026 |
| **Repositorio (fork)** | https://github.com/0-AHCA-0/muestreo-poblacional |

# 1. Objetivos
## Objetivo General: 
Clasificar las variables de la base de fidelización de RetailAndina S.A. con criterio práctico y comparar tres diseños de muestreo frente al parámetro poblacional real, entendiendo por qué la forma de seleccionar los datos importa más que la cantidad de registros.

## Objetivos Específicos:

* Clasificar cada variable de la base de clientes como numérica continua, numérica discreta, categórica nominal o categórica ordinal según su naturaleza y jerarquía.

* Revisar en código los tipos de datos que detectan Python y R por defecto, comprobando la cantidad de categorías únicas y declarando manualmente el orden en las variables ordinales.

* Calcular el ingreso mensual promedio real sobre la población completa de 400 clientes y compararlo con muestras de 20 registros obtenidas por conveniencia, sorteo aleatorio simple y muestreo estratificado.

* Analizar cómo el orden del archivo genera sesgo de selección en el muestreo por conveniencia y cómo cambia el promedio estimado al repetir el sorteo aleatorio con distintas semillas.

# 2. Introducción 
Analizando la base de fidelización de RetailAndina S.A., se aborda primero la clasificación de sus siete variables junto con el código identificador para definir qué operaciones tienen sentido hacer en cada columna antes de armar cualquier reporte comercial. Después, tomando la base completa de 400 clientes cuyo ingreso mensual promedio real es de 1330.72 USD, se estima ese mismo ingreso usando muestras de 20 registros bajo tres métodos distintos: conveniencia, aleatorio simple y estratificado por ciudad.

Realizar esta revisión paso a paso es clave porque los datos mal clasificados llevan directamente a análisis inválidos. Si se confía a ciegas en el tipo de dato que asigna el programa, es fácil cometer el error de promediar una escala de satisfacción del 1 al 5 como si fuera dinero, asumiendo equivocadamente que un nivel 4 representa el doble de conformidad que un nivel 2. Lo mismo ocurre cuando una variable con jerarquía como el nivel educativo se deja como texto común, ya que el sistema la ordena alfabéticamente y coloca Posgrado antes que Primaria, alterando por completo el orden lógico de los reportes.

De igual manera, las pruebas demuestran que una muestra mal diseñada no se corrige agregando más filas. La validez de un estudio depende de cómo se eligen los registros y no solo del tamaño de la muestra. Como el archivo original viene agrupado por ciudad y empieza con los clientes de Quito, tomar los primeros registros por conveniencia deja fuera a Guayaquil y Cuenca. Incluso si se duplicara o quintuplicara la cantidad de filas con ese mismo corte inicial, toda la muestra seguiría concentrada en Quito y el resultado comercial continuaría sesgado.

# 3. Problemas Planteados


## Muestreo de Datos

### muestreo.py

#### Explicación
Para desarrollar el muestreo en Python, primero se carga la base completa de 400 clientes con pandas y se calcula el promedio real de ingresos mensuales, que es de 1330.72 USD, revisando también cuántos clientes pertenecen a Quito, Guayaquil y Cuenca. En el segundo paso se toma una muestra por conveniencia seleccionando únicamente las primeras 20 filas con la función head, lo que sirve para ver qué ocurre cuando se corta un archivo que ya viene ordenado por ciudad. Luego se aplica el muestreo aleatorio simple con la función sample para sortear 20 clientes al azar con igual probabilidad, repitiendo el sorteo con las semillas del 1 al 5 para observar cómo varía el promedio en cada intento. Finalmente, se agrupa la base por ciudad con groupby y se extrae el 5 % de cada grupo para obtener una muestra estratificada de 20 clientes que respeta el peso exacto de cada ciudad, cerrando con una tabla que compara los tres métodos frente al valor real.

In [ ]:
import pandas as pd

pd.set_option("display.width", 100)


def reporte(nombre, muestra, parametro):
    # Muestra el tamano, la media estimada y el conteo por ciudad para detectar sesgos
    n = len(muestra)
    media = muestra["ingresos_mensuales"].mean()
    comp = muestra["ciudad"].value_counts().to_dict()
    print(f"[{nombre}] n={n}  media={media:.2f} USD  "
          f"(parametro {parametro:.2f}, diferencia {media - parametro:+.2f})")
    print(f"    composicion por ciudad: {comp}")


# 1. Poblacion completa y parametro real (N = 400)
poblacion = pd.read_csv("../data/clientes_400.csv")
parametro = poblacion["ingresos_mensuales"].mean()
print(f"Poblacion: {poblacion.shape[0]} clientes x {poblacion.shape[1]} columnas")
print(f"Parametro poblacional - ingreso medio: {parametro:.2f} USD")
print("Ciudades en la poblacion:", poblacion["ciudad"].value_counts().to_dict())
print()

# 2. Muestra por conveniencia: primeras 20 filas del archivo
# Como el CSV esta ordenado por ciudad, este corte selecciona solo clientes de Quito
muestra_conv = poblacion.head(20)

# 3. Muestra aleatoria simple: sorteo equiprobable de 20 clientes (5 % de la base)
muestra_aas = poblacion.sample(20, random_state=1)

# Repeticion del sorteo con semillas 1 a 5 para observar la variabilidad muestral
print("Variabilidad del muestreo aleatorio simple (semillas 1 a 5):")
for semilla in range(1, 6):
    sorteo = poblacion.sample(20, random_state=semilla)
    media_sorteo = sorteo["ingresos_mensuales"].mean()
    print(f"  Semilla {semilla}: media = {media_sorteo:.2f} USD "
          f"(diferencia {media_sorteo - parametro:+.2f})")
print()

# 4. Muestra estratificada: 5 % dentro de cada ciudad manteniendo la proporcion poblacional
# Asegura exactamente 10 de Quito, 6 de Guayaquil y 4 de Cuenca
muestra_est = poblacion.groupby("ciudad", group_keys=False).sample(frac=0.05, random_state=1)

# 5. Comparacion final de los tres disenos frente al parametro
reporte("Conveniencia", muestra_conv, parametro)
reporte("Aleatorio simple", muestra_aas, parametro)
reporte("Estratificado", muestra_est, parametro)
print()

resumen = pd.DataFrame([
    {"diseno": "Conveniencia (head)", "n": len(muestra_conv),
     "media estimada": round(muestra_conv["ingresos_mensuales"].mean(), 2),
     "diferencia": round(muestra_conv["ingresos_mensuales"].mean() - parametro, 2)},
    {"diseno": "Aleatorio simple (sample)", "n": len(muestra_aas),
     "media estimada": round(muestra_aas["ingresos_mensuales"].mean(), 2),
     "diferencia": round(muestra_aas["ingresos_mensuales"].mean() - parametro, 2)},
    {"diseno": "Estratificado (groupby + frac)", "n": len(muestra_est),
     "media estimada": round(muestra_est["ingresos_mensuales"].mean(), 2),
     "diferencia": round(muestra_est["ingresos_mensuales"].mean() - parametro, 2)},
])
print(f"Parametro poblacional: {parametro:.2f} USD")
print(resumen.to_string(index=False))

Poblacion: 400 clientes x 8 columnas
Parametro poblacional - ingreso medio: 1330.72 USD
Ciudades en la poblacion: {'Quito': 200, 'Guayaquil': 120, 'Cuenca': 80}

Variabilidad del muestreo aleatorio simple (semillas 1 a 5):
  Semilla 1: media = 1336.00 USD (diferencia +5.28)
  Semilla 2: media = 1405.00 USD (diferencia +74.28)
  Semilla 3: media = 1235.00 USD (diferencia -95.72)
  Semilla 4: media = 1491.50 USD (diferencia +160.78)
  Semilla 5: media = 1288.00 USD (diferencia -42.72)

[Conveniencia] n=20  media=1348.50 USD  (parametro 1330.72, diferencia +17.78)
    composicion por ciudad: {'Quito': 20}
[Aleatorio simple] n=20  media=1336.00 USD  (parametro 1330.72, diferencia +5.28)
    composicion por ciudad: {'Cuenca': 8, 'Quito': 8, 'Guayaquil': 4}
[Estratificado] n=20  media=1287.50 USD  (parametro 1330.72, diferencia -43.22)
    composicion por ciudad: {'Quito': 10, 'Guayaquil': 6, 'Cuenca': 4}

Parametro poblacional: 1330.72 USD
                        diseno  n  media estimada  

### muestreo.R

#### Explicación
Siguiendo la misma lógica en R, se empieza leyendo el archivo de 400 clientes y calculando la media poblacional de los ingresos mensuales junto con el conteo de clientes por ciudad mediante la función table. Después se arma una función auxiliar para imprimir el tamaño, el promedio y las ciudades presentes en cada muestra. A continuación se extraen las primeras 20 filas con head para el diseño por conveniencia y se realiza el sorteo aleatorio simple de 20 clientes fijando la semilla con set.seed, incluyendo un bucle del 1 al 5 para registrar cómo cambia la media entre sorteos. Por último, se divide la población por ciudad para muestrear el 5 % dentro de cada estrato, asegurando 10 clientes de Quito, 6 de Guayaquil y 4 de Cuenca, y se reúnen las tres estimaciones en un cuadro resumen.

In [1]:
poblacion <- read.csv("../data/clientes_400.csv")

reporte <- function(nombre, muestra, parametro) {
  # Imprime tamano, media estimada y conteo por ciudad para detectar sesgo de seleccion
  media <- mean(muestra$ingresos_mensuales)
  cat(sprintf("[%s] n=%d  media=%.2f USD  (parametro %.2f, diferencia %+.2f)\n",
              nombre, nrow(muestra), media, parametro, media - parametro))
  cat("    composicion por ciudad: ")
  print(table(muestra$ciudad))
}

# 1. Poblacion completa y parametro real (N = 400)
cat(sprintf("Poblacion: %d clientes x %d columnas\n", nrow(poblacion), ncol(poblacion)))
parametro <- mean(poblacion$ingresos_mensuales)
cat(sprintf("Parametro poblacional - ingreso medio: %.2f USD\n", parametro))
cat("Ciudades en la poblacion: ")
print(table(poblacion$ciudad))
cat("\n")

# 2. Muestra por conveniencia: primeras 20 filas del archivo
# El archivo viene ordenado por ciudad, por lo que solo entran clientes de Quito
muestra_conv <- head(poblacion, 20)

# 3. Muestra aleatoria simple: sorteo equiprobable de 20 clientes (5 % del total)
set.seed(1)
muestra_aas <- poblacion[sample(nrow(poblacion), 20), ]

# Repeticion del sorteo con semillas 1 a 5 para ver como varia la estimacion
cat("Variabilidad del muestreo aleatorio simple (semillas 1 a 5):\n")
for (semilla in 1:5) {
  set.seed(semilla)
  sorteo <- poblacion[sample(nrow(poblacion), 20), ]
  media_sorteo <- mean(sorteo$ingresos_mensuales)
  cat(sprintf("  Semilla %d: media = %.2f USD (diferencia %+.2f)\n",
              semilla, media_sorteo, media_sorteo - parametro))
}
cat("\n")

# 4. Muestra estratificada: 5 % dentro de cada ciudad (afijacion proporcional)
# Garantiza 10 clientes de Quito, 6 de Guayaquil y 4 de Cuenca
set.seed(1)
if (requireNamespace("dplyr", quietly = TRUE)) {
  muestra_est <- dplyr::slice_sample(dplyr::group_by(poblacion, ciudad), prop = 0.05)
} else {
  grupos <- split(poblacion, poblacion$ciudad)
  muestra_est <- do.call(rbind, lapply(grupos, function(df) {
    df[sample(nrow(df), size = round(nrow(df) * 0.05)), ]
  }))
}

# 5. Comparacion final de los tres disenos
reporte("Conveniencia", muestra_conv, parametro)
reporte("Aleatorio simple", muestra_aas, parametro)
reporte("Estratificado", muestra_est, parametro)
cat("\n")

resumen <- data.frame(
  diseno = c("Conveniencia (head)", "Aleatorio simple (sample)", "Estratificado (slice_sample)"),
  n = c(nrow(muestra_conv), nrow(muestra_aas), nrow(muestra_est)),
  media_estimada = round(c(mean(muestra_conv$ingresos_mensuales),
                           mean(muestra_aas$ingresos_mensuales),
                           mean(muestra_est$ingresos_mensuales)), 2),
  diferencia = round(c(mean(muestra_conv$ingresos_mensuales) - parametro,
                       mean(muestra_aas$ingresos_mensuales) - parametro,
                       mean(muestra_est$ingresos_mensuales) - parametro), 2)
)
cat(sprintf("Parametro poblacional: %.2f USD\n", parametro))
print(resumen, row.names = FALSE)

Poblacion: 400 clientes x 8 columnas
Parametro poblacional - ingreso medio: 1330.72 USD
Ciudades en la poblacion: 
   Cuenca Guayaquil     Quito 
       80       120       200 

Variabilidad del muestreo aleatorio simple (semillas 1 a 5):
  Semilla 1: media = 1318.00 USD (diferencia -12.72)
  Semilla 2: media = 1287.50 USD (diferencia -43.22)
  Semilla 3: media = 1233.00 USD (diferencia -97.72)
  Semilla 4: media = 1385.50 USD (diferencia +54.78)
  Semilla 5: media = 1283.00 USD (diferencia -47.72)

[Conveniencia] n=20  media=1348.50 USD  (parametro 1330.72, diferencia +17.78)
    composicion por ciudad: 
Quito 
   20 
[Aleatorio simple] n=20  media=1318.00 USD  (parametro 1330.72, diferencia -12.72)
    composicion por ciudad: 
   Cuenca Guayaquil     Quito 
        5         7         8 
[Estratificado] n=20  media=1226.00 USD  (parametro 1330.72, diferencia -104.72)
    composicion por ciudad: 
   Cuenca Guayaquil     Quito 
        4         6        10 

Parametro poblacional: 133

## Clasificación de Variables  

### clasificacion_variables.py

### Explicación

Para clasificar las variables en Python, el primer paso consiste en cargar el archivo de 12 clientes y mostrar sus dimensiones y contenido completo para conocer los datos. En el segundo paso se consulta el atributo dtypes para ver cómo interpreta pandas cada columna de forma automática, notando que la edad, el número de hijos, los ingresos mensuales y la satisfacción se leen como números enteros, mientras que el identificador, el género, el nivel educativo y la ciudad quedan como texto.

En el tercer paso se recorren las columnas con un bucle para listar sus valores únicos, lo que permite descubrir que la variable género tiene tres categorías registradas en la tabla y por eso es nominal en lugar de binaria. En el cuarto paso se corrige el orden de las variables ordinales usando CategoricalDtype para fijar la secuencia desde Primaria hasta Posgrado en educación y del 1 al 5 en satisfacción, comprobando el valor más bajo con min y haciendo además una prueba donde se escribe mal la palabra Postgrado para ver cómo esos registros se vuelven valores vacíos sin dar aviso de error. Al final, se guarda e imprime un resumen claro con el tipo de cada variable.

In [ ]:
import pandas as pd

clientes = pd.read_csv("../data/clientes.csv")

# 1. Estructura general del archivo
print(f"Dimensiones: {clientes.shape[0]} filas x {clientes.shape[1]} columnas\n")
print("Primeras filas:")
print(clientes.to_string(index=False))
print()

# 2. Tipos de datos que infiere pandas por defecto
# Llegan como entero (int64): edad, numero_hijos, ingresos_mensuales y satisfaccion.
# Llegan como texto (object): cliente_id, genero, nivel_educacion y ciudad.
# Aunque satisfaccion aparece como int64, en realidad representa una escala ordinal.
print("Tipos inferidos (dtype):")
print(clientes.dtypes)
print()

# 3. Revision de valores unicos por columna
# Permite ver que genero tiene 3 categorias en la base (Femenino, Masculino, Otro)
print("Valores unicos por variable:")
for columna in clientes.columns:
    unicos = clientes[columna].unique()
    print(f"  {columna} ({len(unicos)} unicos): {unicos.tolist()}")
print()

# 4. Definir el orden correcto en las variables ordinales
orden_educacion = pd.CategoricalDtype(
    categories=["Primaria", "Secundaria", "Universidad", "Posgrado"],
    ordered=True
)
clientes["nivel_educacion"] = clientes["nivel_educacion"].astype(orden_educacion)
print("Nivel educativo minimo:", clientes["nivel_educacion"].min())

orden_satisfaccion = pd.CategoricalDtype(
    categories=[1, 2, 3, 4, 5],
    ordered=True
)
clientes["satisfaccion"] = clientes["satisfaccion"].astype(orden_satisfaccion)
print("Satisfaccion minima registrada:", clientes["satisfaccion"].min())

# Prueba de error de escritura al declarar categorias ("Postgrado" con t)
# Los registros que no coinciden con la lista se vuelven NaN sin lanzar error
educacion_mal_escrita = pd.CategoricalDtype(
    categories=["Primaria", "Secundaria", "Universidad", "Postgrado"],
    ordered=True
)
faltantes_typo = pd.read_csv("../data/clientes.csv")["nivel_educacion"].astype(educacion_mal_escrita).isna().sum()
print(f"Valores perdidos (NaN) al escribir 'Postgrado' con t: {faltantes_typo}")
print()

# 5. Resumen final de la clasificacion de variables
clasificacion = {
    "cliente_id": "Categorica nominal (identificador)",
    "edad": "Numerica continua (registrada en anos cumplidos, discreta en el archivo)",
    "numero_hijos": "Numerica discreta",
    "genero": "Categorica nominal (3 categorias)",
    "nivel_educacion": "Categorica ordinal",
    "ciudad": "Categorica nominal",
    "ingresos_mensuales": "Numerica continua",
    "satisfaccion": "Categorica ordinal",
}

print("Resumen de la clasificacion:")
for variable, tipo in clasificacion.items():
    print(f"  {variable:20s} {tipo}")

| Variable | Tipo | Justificación | Verificación en código |
|---|---|---|---|
| cliente_id | Categórica nominal | Funciona solo como código para identificar a cada cliente, por lo que no mide ninguna cantidad ni tiene sentido promediarla. | Texto con 12 códigos únicos desde C001 hasta C012. |
| edad | Numérica continua | Mide el tiempo de vida, donde sí tiene sentido hablar de medio año, aunque en la tabla se anote en años cumplidos enteros. | Entero con 12 valores únicos entre 22 y 48 años. |
| numero_hijos | Numérica discreta | Proviene de contar personas, así que solo toma números enteros y no existe la mitad de un hijo. | Entero con 5 valores únicos de 0 a 4 hijos. |
| genero | Categórica nominal | Agrupa a los clientes en etiquetas descriptivas sin orden entre sí. No es binaria porque incluye una tercera categoría. | Texto con 3 categorías: Femenino, Masculino y Otro. |
| nivel_educacion | Categórica ordinal | Ordena el grado de estudios siguiendo una jerarquía de avance académico, pero sin una distancia numérica entre niveles. | Texto con 4 niveles ordenados desde Primaria hasta Posgrado, comprobando que el mínimo es Primaria. |
| ciudad | Categórica nominal | Indica la ubicación del cliente mediante nombres de ciudades que no tienen jerarquía ni orden natural. | Texto con 3 valores únicos: Quito, Guayaquil y Cuenca. |
| ingresos_mensuales | Numérica continua | Representa dinero en dólares, donde la mitad del valor y los decimales tienen completo sentido práctico. | Entero con 12 valores únicos entre 760 y 2450 USD. |
| satisfaccion | Categórica ordinal | Califica la conformidad del 1 al 5 con un orden de menor a mayor, pero donde un 4 no significa el doble de satisfacción que un 2. | Llega como entero con 4 niveles presentes del 2 al 5. Se declaró su orden de 1 a 5 comprobando que el mínimo es 2. |

### clasificacion_variables.R

## Explicación 

Para hacer la misma verificación en R, primero se lee el archivo de clientes manteniendo las columnas de texto sin convertir y se revisa la estructura general con la función str para identificar qué variables entran como números enteros y cuáles como caracteres. En el segundo paso se usa un ciclo for con las funciones unique y length para mostrar en pantalla cuántos valores distintos tiene cada columna y cuáles son, confirmando de nuevo las tres categorías de género.

En el tercer paso se transforman las columnas de nivel de educación y satisfacción en factores ordenados mediante la función factor indicando sus niveles de menor a mayor, y se verifica con min que el nivel más bajo sea Primaria y 2 respectivamente. También se incluye la prueba donde se escribe Postgrado con una letra de más para comprobar con is.na cómo el programa convierte esos dos casos en valores faltantes de manera silenciosa, cerrando el script con la impresión del resumen de tipos.

In [ ]:
clientes <- read.csv("../data/clientes.csv", stringsAsFactors = FALSE)

# 1. Estructura general del archivo
cat(sprintf("Dimensiones: %d filas x %d columnas\n\n", nrow(clientes), ncol(clientes)))
print(clientes)
cat("\n")

# 2. Tipos de datos inferidos por R (str)
# Llegan como entero (int): edad, numero_hijos, ingresos_mensuales y satisfaccion.
# Llegan como texto (chr): cliente_id, genero, nivel_educacion y ciudad.
# La columna satisfaccion viene como int pero su naturaleza es categorica ordinal.
str(clientes)
cat("\n")

# 3. Revision de valores unicos por columna
# Confirmamos que genero tiene 3 categorias (Femenino, Masculino, Otro)
cat("Valores unicos por variable:\n")
for (columna in names(clientes)) {
  unicos <- unique(clientes[[columna]])
  cat(sprintf("  %s (%d unicos): %s\n", columna, length(unicos), paste(unicos, collapse = ", ")))
}
cat("\n")

# 4. Definir el orden de las variables ordinales con factores ordenados
educacion_original <- clientes$nivel_educacion

clientes$nivel_educacion <- factor(
  clientes$nivel_educacion,
  levels = c("Primaria", "Secundaria", "Universidad", "Posgrado"),
  ordered = TRUE
)
cat("Nivel educativo minimo:", as.character(min(clientes$nivel_educacion)), "\n")

clientes$satisfaccion <- factor(
  clientes$satisfaccion,
  levels = 1:5,
  ordered = TRUE
)
cat("Satisfaccion minima registrada:", as.character(min(clientes$satisfaccion)), "\n")

# Prueba de error de escritura en los niveles ("Postgrado" con t)
# Los valores que no coinciden con levels pasan a ser NA silenciosamente
educacion_typo <- factor(
  educacion_original,
  levels = c("Primaria", "Secundaria", "Universidad", "Postgrado"),
  ordered = TRUE
)
cat("Valores perdidos (NA) al escribir 'Postgrado' con t:", sum(is.na(educacion_typo)), "\n")

# 5. Resumen final de la clasificacion de variables
clasificacion <- c(
  cliente_id         = "Categorica nominal (identificador)",
  edad               = "Numerica continua (registrada en anos cumplidos, discreta en el archivo)",
  numero_hijos       = "Numerica discreta",
  genero             = "Categorica nominal (3 categorias)",
  nivel_educacion    = "Categorica ordinal",
  ciudad             = "Categorica nominal",
  ingresos_mensuales = "Numerica continua",
  satisfaccion       = "Categorica ordinal"
)

cat("\nResumen de la clasificacion:\n")
for (variable in names(clasificacion)) {
  cat(sprintf("  %-18s %s\n", variable, clasificacion[[variable]]))
}

### Resumen de resultados del muestreo

En este ejercicio la población completa corresponde a los 400 clientes de la base de fidelización, distribuidos en 200 de Quito, 120 de Guayaquil y 80 de Cuenca, y el parámetro poblacional de ingresos mensuales tiene una media real de 1330.72 USD. Aunque aquí la base es pequeña y permite calcular el promedio exacto para usarlo como referencia y evaluar cada método, en el trabajo real de una empresa consultar o encuestar a miles de clientes resulta demasiado caro y lento, por lo que siempre se necesita trabajar con muestras confiables.

| Diseño | n | Media estimada | Composición por ciudad | Diferencia vs. parámetro |
|---|---|---|---|---|
| Conveniencia | 20 | 1348.50 USD | Quito: 20, Guayaquil: 0, Cuenca: 0 | +17.78 USD |
| Aleatorio simple | 20 | 1336.00 USD | Cuenca: 8, Quito: 8, Guayaquil: 4 | +5.28 USD |
| Estratificado | 20 | 1287.50 USD | Quito: 10, Guayaquil: 6, Cuenca: 4 | -43.22 USD |

Al repetir cinco veces el sorteo del muestreo aleatorio simple en Python cambiando la semilla del 1 al 5, las medias obtenidas fueron 1336.00 USD en la semilla 1, 1405.00 USD en la semilla 2, 1235.00 USD en la semilla 3, 1491.50 USD en la semilla 4 y 1288.00 USD en la semilla 5.

## Preguntas Planteadas

**P1.** ¿Qué te dijo el dtype de `satisfaccion` y por qué no basta para clasificarla? ¿Qué pasa si una ordinal queda como texto sin declarar el orden?

Revisando los tipos que detecta el programa por defecto, se observa que la variable satisfacción entra como número entero simplemente porque en el archivo fue guardada con números del 1 al 5. Sin embargo, ese dato técnico no alcanza para clasificarla como numérica, ya que esos números solo son etiquetas de opinión: un cliente con nivel 4 está más conforme que uno con nivel 2, pero no siente el doble de satisfacción, así que promediarlos directamente como si fueran dólares daría una idea equivocada. Por otro lado, si una variable con jerarquía como el nivel de educación se deja como texto común sin declarar su orden, el programa ordena las categorías alfabéticamente y pone Posgrado antes que Primaria y Secundaria, dañando el sentido de las tablas, los gráficos y el cálculo del mínimo. Además, al declarar los niveles a mano hay que escribirlos igual que en la base, porque un pequeño error como escribir Postgrado con t hace que esos dos registros no coincidan con la lista y se pierdan como valores vacíos sin que el programa avise.

**P2.** ¿Cuánto vale el parámetro poblacional y cuánto estimó cada diseño? ¿Cuál quedó más cerca y por qué eso **no** alcanza para declarar un ganador?

Comparando los resultados frente al parámetro poblacional real de 1330.72 USD, con la semilla 1 en Python el muestreo por conveniencia estimó 1348.50 USD, el aleatorio simple dio 1336.00 USD y el estratificado marcó 1287.50 USD. Aunque en esta corrida el aleatorio simple quedó a solo 5.28 USD del valor real, e incluso el corte por conveniencia estuvo más cerca que el estratificado, mirar un único resultado no sirve para declarar un ganador porque en muestras pequeñas de 20 personas pesa mucho la suerte del sorteo. Basta con pasar a la semilla 4 en el mismo muestreo aleatorio simple para que el promedio suba hasta 1491.50 USD, alejándose por más de 160 USD del valor real. Por eso un diseño de muestreo se evalúa por qué tan bien representa a los grupos de la población y por evitar sesgos, no por haber acertado de casualidad en un solo intento.

**P3.** La muestra por conveniencia también tiene 20 filas. ¿Qué defecto tiene y cómo lo delatas en la composición por ciudad? ¿Se corrige tomando más filas?

Observando el conteo por ciudad, el problema de la muestra por conveniencia salta a la vista: los 20 clientes elegidos son todos de Quito y no aparece nadie de Guayaquil ni de Cuenca. Esto pasa porque el archivo viene ordenado por ciudad y los primeros 200 registros pertenecen a Quito, que justamente es la plaza con el ingreso promedio más alto con 1390.50 USD, por encima de Guayaquil con 1322.58 USD y Cuenca con 1193.50 USD. Debido a este orden físico en la tabla, el defecto no se arregla tomando más filas con el mismo método: aunque se tomaran los primeros 50, 100 o 200 clientes, toda la muestra seguiría siendo de Quito y seguiría inflando el ingreso promedio real de la cadena.

**P4.** El sorteo aleatorio dio un número distinto en cada semilla. ¿Qué implica esa variabilidad para una empresa que va a decidir con un solo estudio?

Analizando los cinco sorteos del muestreo aleatorio simple, se ve que el promedio estimado baja hasta 1235.00 USD en la semilla 3 y sube hasta 1491.50 USD en la semilla 4, lo que deja una diferencia de 256.50 USD usando exactamente el mismo procedimiento al azar. Para una empresa que va a tomar una decisión comercial guiándose por un solo estudio, esta variación significa que un promedio sacado con apenas 20 clientes puede moverse bastante hacia arriba o hacia abajo por puro azar. Por lo tanto, la gerencia no debe tomar ese número como un dato exacto, sino acompañarlo de un margen de error o pedir una muestra más grande si necesita precisión antes de definir precios o presupuestos.

**P5.** ¿Cuándo usarías cada diseño en la práctica? Relaciona cada uno con un uso real (piloto, subgrupos que deben aparecer, conclusión sobre toda la población).

Evaluando el uso práctico de cada método dentro de la empresa, el muestreo por conveniencia solo conviene usarlo en pruebas piloto rápidas, como revisar que el código funcione bien o probar con los primeros clientes disponibles si las preguntas de una encuesta se entienden, pero nunca para sacar conclusiones sobre toda la clientela. Por su parte, el muestreo aleatorio simple funciona bien cuando se busca una conclusión general sobre toda la base, se tiene la lista completa para sortear sin favoritismos y los clientes son bastante parecidos entre sí. En cambio, el muestreo estratificado es el que se necesita cuando la población está dividida en subgrupos importantes con comportamientos distintos, como las ciudades donde vende RetailAndina, ya que obliga a que cada ciudad entre en su proporción exacta de 10 clientes de Quito, 6 de Guayaquil y 4 de Cuenca, evitando que el azar sobrepondere una ciudad pequeña como pasó en la semilla 1 del sorteo simple donde Cuenca tuvo 8 de los 20 clientes.

# 4. Conocimientos Adquiridos en la Práctica 
A lo largo de la práctica se comprendió cómo distinguir entre el tipo de dato que asigna automáticamente el lenguaje de programación y la clasificación estadística real de una variable según su continuidad y su orden lógico. También se aprendió a configurar manualmente el orden jerárquico de las variables ordinales tanto en Python como en R para que los reportes respeten su secuencia real y para detectar cuándo un error de escritura transforma registros válidos en datos faltantes. Finalmente, se puso en práctica la diferencia entre tomar datos por conveniencia, hacer un sorteo aleatorio simple controlando la semilla de repetición y armar un muestreo estratificado que conserve el peso real de cada ciudad frente al parámetro poblacional.

# 5. Declaración de Uso de IA
Yo, Aidan Carrasco, declaro haber usado Inteligencia Artificial, especificamente Gemini para una redacción coherente del informe. El codigo fue hecho por mi persona con ayuda de lo que se realizó en clases y adaptado a los ejercicios correspondientes.

# 6. Conclusiones 

Analizando cada una de las variables y sus sesgos, se observa que no basta con el tipo de dato que el código detecta por defecto. Variables como la satisfacción o el nivel de educación tienen una jerarquía que debe definirse desde el inicio para evitar que el sistema las ordene alfabéticamente o genere valores faltantes sin darnos cuenta. Del mismo modo, al tomar solo las primeras filas por comodidad se produjo un sesgo directo al seleccionar únicamente clientes de Quito por cómo venía ordenado el archivo, demostrando que aumentar filas bajo ese mismo método no soluciona el problema de selección.

Revisando los resultados para RetailAndina, se nota que el sorteo puramente aleatorio provoca variaciones muy marcadas, haciendo que el promedio cambie entre 1235.00 y 1491.50 USD frente al valor real de 1330.72 USD. Por esta razón, el muestreo estratificado es la alternativa más adecuada para respaldar las decisiones de la gerencia, ya que controla esa variabilidad y asegura la proporción exacta de cada ciudad con 10 clientes de Quito, 6 de Guayaquil y 4 de Cuenca.

# 7. Bibliografía

* Pandas Development Team. (2026). *pandas documentation: Categorical data & Indexing and selecting data (sample)*. https://pandas.pydata.org/docs/
* R Core Team. (2026). *R: A Language and Environment for Statistical Computing*. R Foundation for Statistical Computing. https://www.R-project.org/
* Wickham, H., François, R., Henry, L., Müller, K., & Vaughan, D. (2026). *dplyr: A Grammar of Data Manipulation (slice_sample)*. https://dplyr.tidyverse.org/